<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Your Places, Week 5 Session 2: Your Places as a Table

### CP101: Introduction to Urban Data Analytics
#### Wednesday, Dictionaries and Pandas I

Last week you used Python's `csv` module to read a places file. Each row was a list of strings.
You selected values by position and used loops to count or filter the rows.

Today you will use **dictionaries** to store values under names, then use **pandas** to work with
a table. A pandas table is called a **DataFrame**. You can select its columns by name and apply
an operation to a whole column at once.

We will use `combined_places.csv`, which contains fifteen places from three lists of five.
You will count categories, filter places by rating, extract parts of addresses, and make charts.
Finally, you will use latitude and longitude to estimate each place's distance from Doe Library.


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 1. A dictionary, then a table

With `csv.reader`, each row is a list. To get a value, you use its position: `row[0]` is the name,
`row[1]` is the address, and so on. The header tells you what each position means.

Below are the header and first row of `combined_places.csv`, typed as two lists.
Python counts positions from zero. Find `rating` in the header: which position holds its value?


In [ ]:
# the header of combined_places.csv, then its first row, both as plain lists
header = ["name", "address", "lat", "lon", "category", "notes", "rating", "accessibility"]
row = ["Doe Library", "Doe Library, Campanile Way, Berkeley, CA 94720", "37.8724", "-122.2592",
       "reading", "Quiet north reading room, best for long sessions", "love it", ""]

print(row)
print()
print("rating:", row[6])


The rating is at position `6`, so `row[6]` returns `"love it"`. This works as long as you know
the column order.

A **dictionary** stores values under **keys**. In the dictionary below, `"rating"` is a key and
`"love it"` is its value. You can read that value with `place["rating"]`.

The `.items()` method provides each key and its value together. In the loop, `key` receives the
key and `value` receives the corresponding value. How many times will the loop run?


In [ ]:
place = {
    "name": "Doe Library",
    "category": "reading",
    "lat": 37.8724,
    "lon": -122.2592,
    "rating": "love it",
}

print(place)
print("category:", place["category"])

# .items() provides one key-value pair per iteration
for key, value in place.items():
    print(key, "->", value)


### Build a dictionary from two lists

The lists below describe eight places. Items at the same position belong together: the first
name, category, and rating all describe Doe Library.

To find Top Dog's rating using the lists, first find its position in `place_names`, then use that
position in `place_ratings`.

We can also build a dictionary with names as keys and ratings as values. `zip()` pairs items at
the same position in the two lists. For example, its first pair is `("Doe Library", "love it")`.
`dict()` turns those pairs into dictionary entries.

Both approaches below should return the same rating. Run the cell and compare them.


In [ ]:
# The example collection: three lists, lined up by position
place_names = ["Doe Library", "Yogurt Park", "Games Of Berkeley", "Memorial Glade",
               "Top Dog", "Moe's Books", "Pegasus Books", "Noodle Dynasty"]
place_categories = ["reading", "dessert", "gaming", "relaxing",
                    "food", "bookstore", "bookstore", "food"]
place_ratings = ["love it", "like it", "love it", "just okay",
                 "love it", "like it", "love it", "just okay"]

# by position: find where Top Dog sits, then read the other list at that spot
where = place_names.index("Top Dog")
print("by position:", place_ratings[where])

# by label: use each name as a key and its rating as the value
rating_by_name = dict(zip(place_names, place_ratings))
print("by label   :", rating_by_name["Top Dog"])

print("keys:", len(rating_by_name))


Both lookups return `"love it"`. Once the dictionary is built, you can look up a rating directly
by name. When building it with `zip()`, the two lists must be correctly aligned; `zip()` stops
when the shorter list ends.

You can inspect a dictionary in three ways:

- `.keys()` gives its keys, such as `"Doe Library"`.
- `.values()` gives its values, such as `"love it"`.
- `.items()` gives key-value pairs, such as `("Doe Library", "love it")`.

**Each key can appear only once.** Assigning a value to an existing key replaces its previous
value. The next cell changes Doe Library's rating. Will that change the number of keys?

This matters when combining people's lists: if two records have the same place name, a dictionary
keyed by name can keep only one value under that name.


In [ ]:
print("labels:", list(rating_by_name.keys())[:3], "...")
print("values:", list(rating_by_name.values())[:3], "...")
print("pairs :", list(rating_by_name.items())[:2], "...")

# Doe Library is already a key, so this replaces its rating rather than adding an entry
rating_by_name["Doe Library"] = "just okay"
print()
print("Doe Library is now:", rating_by_name["Doe Library"])
print("keys:", len(rating_by_name), "(unchanged)")


### Build a table from a dictionary

A **DataFrame** is a table with rows and columns. One way to build it is to pass a dictionary to
`pd.DataFrame()`:

- Each key becomes a column name.
- Each value is a list containing that column's data.
- Items at the same position in the lists form one row.

The column lists must have the same length. Here, five lists of five values produce a table with
five columns and five rows. Which columns contain numbers?


In [ ]:
import pandas as pd

places = pd.DataFrame({
    "name":     ["Doe Library", "Yogurt Park", "Games Of Berkeley", "Memorial Glade", "Mezzo"],
    "category": ["reading", "dessert", "gaming", "relaxing", "food"],
    "lat":      [37.8724, 37.8680, 37.8677, 37.8733, 37.8663],
    "lon":      [-122.2592, -122.2598, -122.2584, -122.2594, -122.2588],
    "rating":   ["love it", "like it", "love it", "just okay", "love it"],
})
places


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 2. Counting with a dictionary

How many places are in each category?

We can use a dictionary to store the counts. Each **key** is a category, and its **value** is the
number of places counted so far. For example, `{"food": 2, "bookstore": 2}` means we have counted
two food places and two bookstores.

Start with an empty dictionary. Then loop through `place_categories`, adding one to the appropriate
count each time. The loop below does this in two steps: look up the current count, then store that
count plus one.

`category_counts.get(category, 0)` looks up the current count. If the category is not in the
dictionary yet, it returns `0`. The assignment on the next line creates the entry or updates its
existing count. Reading a missing key with `category_counts[category]` would raise a `KeyError`;
assigning to it creates the key.

**Before you run the code:** look back at `place_categories`. Which categories should have a count
of `2`?


In [ ]:
category_counts = {}

for category in place_categories:
    current_count = category_counts.get(category, 0)
    category_counts[category] = current_count + 1

print(category_counts)
print()
print("places counted:", sum(category_counts.values()), "of", len(place_categories))


`food` and `bookstore` each have a count of `2`. The first time the loop reaches `"bookstore"`,
`.get()` returns `0`, and the assignment stores `1`. The next time, `.get()` returns `1`, and the
assignment stores `2`.

You can also combine the two steps inside the loop into one line:

```python
category_counts[category] = category_counts.get(category, 0) + 1
```

Notice that `sport` is missing from the result: it never appears in `place_categories`, so the
loop never creates that key. The dictionary contains only categories found in the list.

The total is `8`, matching the number of places in the list. Comparing the sum of the counts with
the length of the list is a useful check: a mismatch means something was missed or counted extra.

In `pandas`, `.value_counts()` does this counting for us. The next cell builds a table from the
three lists above, then counts the values in its `category` column.

Compare the counts with your dictionary. Are the categories listed in the same order?


In [ ]:
example_places = pd.DataFrame({
    "name":     place_names,
    "category": place_categories,
    "rating":   place_ratings,
})

print("rows:", len(example_places))
example_places["category"].value_counts()


The two methods give the same counts. `.value_counts()` lists the most frequent categories first.
The dictionary lists categories in the order they were first encountered in `place_categories`.

The loop shows how to keep a running count. When working with a pandas column, you can use
`.value_counts()` to get those counts directly.


<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: count the ratings

1. Build `rating_counts` by looping through `place_ratings`. Use the two-step counting pattern
   from section 2, then compare your result with `example_places["rating"].value_counts()`.
2. Build `names_by_category`, where each key is a category and each value is a list of place names.
   Loop over `zip(place_names, place_categories)` to get each name and its category together.
   For a new category, start with an empty list. Add each name as a one-item list, `[name]`.

<details>
<summary><b>Check your answer</b></summary>

The rating counts are `"love it": 4`, `"like it": 2`, and `"just okay": 2`.

For the second task, the value under each key is a list. Inside the loop, use:

```python
current_names = names_by_category.get(category, [])
names_by_category[category] = current_names + [name]
```

The `"bookstore"` entry should contain `"Moe's Books"` and `"Pegasus Books"`.

</details>


In [ ]:
# Your code here


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 3. Read a table from a file

So far, we have typed our example data into Python. Now we will read it from a CSV file.
`combined_places.csv` contains fifteen places and eight columns. It combines three people's
lists of five places. Next week, you will combine your team's files yourselves.

`pd.read_csv()` reads the file and creates a DataFrame. By default, it uses the first line as the
column names. It also recognises quoted fields: an address such as
`"2433 Durant Ave A, Berkeley, CA 94704"` stays in one column even though it contains commas.
Splitting the file's lines with `.split(",")` would split that address too.

The next cell stores the table in `places`, replacing the small table we typed earlier.
Run it and find Yogurt Park's address. Is the full address in one cell?


In [ ]:
places = pd.read_csv("../data/raw/combined_places.csv")
places


Pandas also infers a **data type** for each column from the values in the file. For example,
`lat` contains decimal numbers, while `rating` contains words.

`places.dtypes` shows the type assigned to each column. Run it and compare `lat` with `rating`.


In [ ]:
places.dtypes


`lat` and `lon` have type `float64`, which stores numbers with decimal places.
The text columns have type `object` in our pandas environment. `object` can hold several kinds
of Python values; in columns such as `name` and `rating`, those values are strings.
If you use pandas 3, these text columns may be labelled `str` instead.

Every `accessibility` entry is blank. With the default CSV settings, pandas reads these blanks
as missing values (`NaN`) and gives this column type `float64`. That does not tell us what kind
of information should eventually go in the column.

Type inference depends on the file's contents. If a latitude contains text such as `"37.8 degrees"`,
pandas may read the column as text. Check `.dtypes` after loading data, especially for columns
you plan to use in calculations.


<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: add your place

1. Open `../data/raw/combined_places.csv` and add a row with the same eight fields as the header.
   Keep latitude and longitude numeric, and put quotes around any field containing a comma.
2. Save the file, then re-run the two code cells in section 3. Does your place appear? Are `lat`
   and `lon` still numeric?

`pd.read_csv()` reads the saved file, so save your edits before re-running it. You can restore the
supplied data from `../data/raw/combined_places_backup.csv` if needed.

The example counts and answers below use the original fifteen rows. If you keep your added row,
some results will change.

<details>
<summary><b>Check your answer</b></summary>

After adding one row, the table should contain sixteen rows. In this file, `lat` and `lon` should
still have type `float64`. If either becomes a text column, check the values you added for letters,
degree symbols, or other extra characters.

</details>


In [ ]:
# Your code here


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 4. Count the ratings

The ratings have an order: `"just okay"`, then `"like it"`, then `"love it"`. Data with categories
in a meaningful order is called **ordinal data**.

These labels do not tell us how large the difference between two ratings is. Assigning them
numbers would require us to decide what those differences mean before interpreting an average.
Calling `.mean()` on the text ratings raises a `TypeError` because they are not numeric.

We can describe the ratings by counting how many places have each one. Use `.value_counts()`
again, this time on `places["rating"]`. Which rating is most common?


In [ ]:
places["rating"].value_counts()


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 5. Filter rows with a condition

Which places have the rating `"love it"`?

`places["rating"] == "love it"` compares each value in the rating column with `"love it"`.
It returns `True` for a match and `False` otherwise. This column of Boolean values is called a
**mask**.

The next cell stores the mask as `is_favourite`. Compare its `True` entries with the count of
`"love it"` from section 4.


In [ ]:
is_favourite = places["rating"] == "love it"
is_favourite


Use the mask inside square brackets to select rows: `places[is_favourite]` keeps the rows where
the mask is `True`.

We store the filtered table as `favourites`. Check its ratings, then look at the row labels down
the left side. Do they still run from zero without gaps?


In [ ]:
favourites = places[is_favourite]
favourites


### Select a row by position or label

In the supplied data, `favourites` has row labels `0, 2, 4, 5, 7, 11, 12`. Filtering preserves
the original row labels, so the second row has **position `1`** and **label `2`**.

Pandas provides a different way to select each:

- `.iloc[1]` selects the row at position `1`, counting from zero.
- `.loc[2]` selects the row whose label is `2`.

Both expressions below select the same place. Use the displayed table to identify it before
running the cell.


In [ ]:
print("second favourite, by position:", favourites.iloc[1]["name"])
print("favourite labelled 2, by label :", favourites.loc[2, "name"])


Both lines return `"Games Of Berkeley"`. It is the second row in `favourites` and has label `2`.

There is no row labelled `1` in this filtered table, so `favourites.loc[1, "name"]` would raise
a `KeyError`. The second argument to `.loc` selects the column; here it is `"name"`.

To give a filtered table new row labels starting at zero, use:

```python
favourites = favourites.reset_index(drop=True)
```

`drop=True` leaves out the old labels. Without it, they become an additional column.


### Combine two conditions

Now select places that are rated `"love it"` **and** are north of latitude `37.869`, our approximate
reference for Bancroft Way. We already have the first mask. The next cell creates the second.

Combine masks with:

- `&` to keep rows where **both** conditions are true.
- `|` to keep rows where **at least one** condition is true.

The Python words `and` and `or` operate on single truth values. Using them with these pandas
masks raises a `ValueError`; use `&` and `|` for comparisons row by row.

You can combine named masks, as in `is_favourite & is_north_of_bancroft`, or write both comparisons
inside the brackets. In the second form, put parentheses around **each comparison** so Python
evaluates it before combining the masks.

Compare the two counts of places meeting both conditions. They should agree and cannot exceed
the count for either condition alone.


In [ ]:
is_north_of_bancroft = places["lat"] > 37.869     # approximate latitude of Bancroft Way

print("favourites:", is_favourite.sum())
print("north of Bancroft:", is_north_of_bancroft.sum())
print("both, named masks    :", len(places[is_favourite & is_north_of_bancroft]))
print("both, written in full:",
      len(places[(places["rating"] == "love it") & (places["lat"] > 37.869)]))

# Parentheses group each comparison before & combines them.
places[(places["rating"] == "love it") & (places["lat"] > 37.869)]


A mask also lets you count matches without displaying the filtered rows. `.sum()` adds its Boolean
values, treating `True` as `1` and `False` as `0`. For example, `is_favourite.sum()` counts places
rated `"love it"`.


<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: filter by category

Create `is_reading = places["category"] == "reading"`, then use it to select the reading places.
How many rows match? What are their row labels?

<details>
<summary><b>Check your answer</b></summary>

`is_reading` is the mask. `places[is_reading]` is the filtered table. In the supplied data, one row
matches: Doe Library, with its original label `0`. `is_reading.sum()` returns `1`.

</details>


In [ ]:
# Your code here


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 6. Extract information from text

The `address` column contains several pieces of information in one string. For example,
`"2433 Durant Ave A, Berkeley, CA 94704"` includes a street address, city, state, and ZIP code.

Pandas has already read this address as one field. Now we can use `.split(", ")` on the address
itself to separate it at each comma followed by a space.

The next cell uses Yogurt Park's address. How many pieces will the split produce? Which piece
contains the ZIP code?


In [ ]:
address = places.loc[1, "address"]     # the row labelled 1: Yogurt Park
print(address)

pieces = address.split(", ")           # a two-character separator: the comma and the space
print(pieces)
print("pieces:", len(pieces))

print("last piece:", pieces[-1])       # -1 counts from the end, exactly as it does for a list
print("ZIP code  :", address[-5:])     # a string slices like a list: its last five characters


The result is a list of three strings: `"2433 Durant Ave A"`, `"Berkeley"`, and `"CA 94704"`.
`pieces[-1]` selects the last one. `address[-5:]` selects the last five characters of the original
string, giving `"94704"`.

These operations depend on the format of the address. For this file, the ZIP code is always at
the end. An address with a ZIP+4 code or a country name at the end would need different handling.

The separator also matters when splitting words:

- `.split(" ")` splits at each individual space, keeping empty strings between adjacent spaces.
- `.split()` splits on runs of whitespace and leaves out empty pieces.

Run the next cell and compare the resulting lists.


In [ ]:
untidy = "  Durant   Ave "
print(untidy.split(" "))    # split at each space; keep empty strings
print(untidy.split())       # split on runs of whitespace; omit empty strings


### Apply text operations to a column with `.str`

So far, we have worked with one address. Use `.str` to apply a text operation to each value in
a pandas column. For example, `places["address"].str.split(", ")` returns one list of pieces
for each row.

We can extract the ZIP codes in steps:

1. Split each address at `", "`.
2. Take the last piece, such as `"CA 94704"`.
3. Split that piece on whitespace and take its last item, `"94704"`.

The next cell names each intermediate result, then stores the ZIP codes in a new `zip` column.
It also checks whether taking the last five characters gives the same answer for every row.


In [ ]:
address_parts = places["address"].str.split(", ")
state_and_zip = address_parts.str[-1]
zips = state_and_zip.str.split().str[-1]
print(zips.tolist())

# Check whether the last five characters give the same ZIP code for every row.
print("same as the slice?", (places["address"].str[-5:] == zips).all())

places["zip"] = zips
places["zip"].value_counts()


The supplied file contains four ZIP codes: ten rows have `94704`, three have `94720`, and one each
has `94708` and `94705`.

Extracting a ZIP code from an address does not verify that the address is correct. For example,
Memorial Glade's address contains `94708`; we would need to check the address against a reliable
source before using that ZIP code to locate it. The new column preserves the information in the
original text, including any errors.


### Filter rows using text

Which addresses contain `"Durant"`?

`.str.contains("Durant")` tests each address and returns a mask. We can count its `True` values
with `.sum()`, then use `.loc` to display the matching rows and selected columns.


In [ ]:
on_durant = places["address"].str.contains("Durant")
print("on Durant:", on_durant.sum())

places.loc[on_durant, ["name", "address", "rating"]]


`.str.contains("Durant")` is case sensitive by default: `"Durant"` matches, but `"durant"` does not.
It can match part of a word, so a search for `"Ave"` would also match `"Avenue"`.

To ignore capitalisation, we can first use `.str.lower()`. The next example finds names starting
with `"caf"`, which matches both `"Café Milano"` and `"Caffe Strada"` after lowercasing.

Lowercasing does not remove accents or make different spellings equal. `"Café"` and `"Cafe"` are
different strings. The example also compares the length of `"Café"` as a string with the length
of its UTF-8 representation: four characters use five bytes because `é` uses two bytes.


In [ ]:
is_cafe = places["name"].str.lower().str.startswith("caf")
print(places.loc[is_cafe, "name"].tolist())

print("Café == Cafe ?", "Café" == "Cafe")
print("characters:", len("Café"), "   bytes in UTF-8:", len("Café".encode("utf-8")))


<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: inspect the addresses

1. Build a mask called `on_telegraph` from the `address` column. Show the names and ratings of
   places whose address contains `"Telegraph"`. Compare the count with the count for Durant.
2. Extract the first part of each address with `places["address"].str.split(", ").str[0]`.
   Store it in `address_start`, then assign it to `places["address_start"]` and display it beside
   `name`. Which entries do not begin with a street number?

<details>
<summary><b>Check your answer</b></summary>

In the supplied data, four addresses contain `"Telegraph"` and three contain `"Durant"`.

The first parts for Doe Library, Memorial Glade, Little Gem Belgian Waffles, and Willard Park
have no street number. Doe Library's begins with a building name. This shows why the first part
of an address is not always a numbered street address. Missing numbers alone do not tell us
whether an address is incorrect; those entries would need further checking.

</details>


In [ ]:
# Your code here


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 7. Plot counts and locations

A bar chart lets us compare the number of places in each category. We will use **Plotly**, which
adds details you can see by hovering over a bar.

First, `.value_counts()` counts the categories. `.reset_index()` turns the result into a table,
and we name its columns `category` and `places`.

`px.bar()` uses `places` for bar length and `category` for the labels. The remaining lines set
colours, add count labels, and put the largest count at the top. Compare the longest bar with
the counts in the table.


In [ ]:
import plotly.express as px

# Convert the category counts to a table with named columns.
counts = places["category"].value_counts().reset_index()
counts.columns = ["category", "places"]

# Give the largest count an orange bar and the others grey bars.
accent = ["#E8722C" if n == counts["places"].max() else "#B4B4AE" for n in counts["places"]]

fig = px.bar(counts, x="places", y="category", orientation="h", text="places",
             title="Places per category", height=400)
fig.update_traces(marker_color=accent, marker_cornerradius=4,
                  textposition="outside", cliponaxis=False, texttemplate="%{text:.0f}",
                  hovertemplate="%{y}: %{x} places<extra></extra>")
fig.update_layout(plot_bgcolor="white", bargap=0.35, showlegend=False,
                  yaxis_title=None, margin=dict(l=10, r=30, t=60, b=10))
fig.update_yaxes(categoryorder="total ascending")
fig.update_xaxes(visible=False)   # the bar labels show the counts
fig


The chart uses three choices to make the counts easier to read:

- The bars are sorted by count, so nearby bars are easy to compare.
- The largest count is highlighted in orange; the other bars are grey.
- Each bar has a count label, so we can read exact values without an x-axis scale.

A scatter plot can show where the places are relative to one another. We put longitude on the
x-axis and latitude on the y-axis. Each row becomes one point, and hovering reveals its name,
category, rating, and ZIP code.

Find Doe Library on the plot. Which places are north of it, and which are south?


In [ ]:
fig = px.scatter(places, x="lon", y="lat",
                 hover_name="name", hover_data=["category", "rating", "zip"],
                 title="Place locations by longitude and latitude",
                 height=430)
fig.update_traces(marker=dict(size=11, color="#00548F",
                              line=dict(width=1.5, color="white")))
fig.update_layout(plot_bgcolor="white", margin=dict(l=10, r=30, t=60, b=10))
fig.update_xaxes(title="longitude", showgrid=True, gridcolor="#EAEAE6", zeroline=False)
fig.update_yaxes(title="latitude", showgrid=True, gridcolor="#EAEAE6", zeroline=False)
fig


In this plot, a higher latitude appears farther up, and a higher longitude appears farther right.
In Berkeley, these directions correspond to north and east. Remember that `-122.25` is greater
than `-122.26`, so it lies farther east.

In the supplied data, Half Price Books is the leftmost point and Willard Park is the lowest.
Hover over them to check their names and coordinates.

The axes show degrees. A degree of latitude and a degree of longitude represent different distances
here, and the plot does not adjust its proportions for that difference. We should calculate
distances from the coordinates instead of measuring the gaps on the screen.


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 8. Estimate distance from Doe Library

How far is each place from Doe Library in a straight line?

For nearby places, we can estimate the distance by converting coordinate differences to metres.
We use two approximate conversion factors:

- One degree of latitude is about `111,000` metres.
- One degree of longitude is about `111,000 × cos(latitude)` metres, or roughly `87,600` metres
  at Berkeley's latitude. Longitude lines get closer together toward the poles.

The calculation has three steps:

1. Select Doe Library's row as the reference location.
2. Subtract its longitude and latitude from each place's coordinates, then convert those
   differences to metres. These give the east-west and north-south offsets.
3. Use the Pythagorean theorem: distance is the square root of `east² + north²`.

Subtracting one number from a pandas column applies the subtraction to every row. This is an
example of a **vectorized operation**: we write one column calculation instead of a Python loop.

In the code, `math.radians()` converts latitude from degrees to the units needed by `math.cos()`.
`** 2` squares a value, and `** 0.5` takes its square root. The underscore in `111_000` is just a
digit separator; it has the same value as `111000`.

Which place should have distance zero? Run the cell and check the first row of the sorted result.


In [ ]:
import math

library = places.loc[places["name"] == "Doe Library"].iloc[0]   # one row, as a Series

m_per_degree_lat = 111_000
m_per_degree_lon = 111_000 * math.cos(math.radians(library["lat"]))
print("metres per degree of latitude :", m_per_degree_lat)
print("metres per degree of longitude:", round(m_per_degree_lon))

east  = (places["lon"] - library["lon"]) * m_per_degree_lon    # a whole column at once
north = (places["lat"] - library["lat"]) * m_per_degree_lat
places["distance_m"] = ((east ** 2 + north ** 2) ** 0.5).round()

places.sort_values("distance_m")[["name", "category", "rating", "distance_m"]]


Doe Library has distance zero because it is the reference point. In the supplied data,
Caffe Strada is about `509` metres away, Top Dog is about `524` metres away, and Willard Park is
the farthest at about `1,253` metres.

These are approximate **straight-line distances**. The calculation treats the local area as flat
and uses the same conversion factors for all rows. It is useful for nearby places, but it does
not measure a walking route or account for streets, entrances, or barriers. Rounding to whole
metres changes how the result is displayed, not the accuracy of the estimate.

`.sort_values("distance_m")` returns a sorted table and leaves `places` in its original order.
To keep the sorted result for later use, assign it to a variable, for example:

```python
places_by_distance = places.sort_values("distance_m")
```


We can also compare distances with a bar chart. Each bar represents one place, and its length is
the estimated distance in metres. The labels show the distance directly.

The next cell leaves out Doe Library so the chart focuses on distances to the other places.
It sorts them by distance and highlights the farthest one. Check that this is the same place
as the last row of the sorted table.


In [ ]:
# Sort distances in ascending order; the horizontal chart puts the last row at the top.
from_library = places[places["name"] != "Doe Library"].sort_values("distance_m")

accent = ["#E8722C" if d == from_library["distance_m"].max() else "#B4B4AE"
          for d in from_library["distance_m"]]

fig = px.bar(from_library, x="distance_m", y="name", orientation="h", text="distance_m",
             title="Straight-line distance from Doe Library", height=480)
fig.update_traces(marker_color=accent, marker_cornerradius=4,
                  textposition="outside", cliponaxis=False, texttemplate="%{text:,.0f} m",
                  hovertemplate="%{y}: %{x:,.0f} m<extra></extra>")
fig.update_layout(plot_bgcolor="white", bargap=0.35, showlegend=False,
                  yaxis_title=None, margin=dict(l=10, r=50, t=60, b=10))
fig.update_xaxes(visible=False)
fig


<hr style="border: 1px solid #fdb515;" />

### ✏️ Your Turn: use the distance column

1. **Within 800 metres.** Using the distances from Doe Library, create a mask for
   `places["distance_m"] <= 800`. Count the matching rows, then combine this mask with
   `places["rating"] == "love it"` to show only favourites within that distance.
   These are straight-line distances, so the result does not establish a walking time.
2. **Choose another reference.** Copy the distance calculation into the cell below and use
   Willard Park as the reference instead of Doe Library. Recompute the longitude conversion
   factor using Willard Park's latitude. Exclude Willard Park itself, sort by the new distance,
   and find the nearest and farthest places.

<details>
<summary><b>Check your answer</b></summary>

For the supplied data, twelve places are within `800` metres of Doe Library, including the library
itself at zero metres. Five of those twelve are rated `"love it"`. To exclude the reference from
the count, also require `places["name"] != "Doe Library"`.

From Willard Park, the nearest other place is Moe's Books, about `515` metres away, and the farthest
is Half Price Books, about `1,545` metres away. After filtering out the reference and sorting, use
`.iloc[0]` for the nearest remaining row and `.iloc[-1]` for the farthest.

</details>


In [ ]:
# Your code here


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Going deeper (optional)

### Look up a key that may be missing

In section 2, `.get(category, 0)` returned zero when a category had not been counted yet.
The same method works for other dictionary lookups.

The `place` dictionary from section 1 has a `category` key but no `notes` key. Compare these cases:

- `.get("category")` returns the stored category.
- `.get("notes")` returns `None` because the key is missing and no default was supplied.
- `.get("notes", "no notes written")` returns the default text.

The expression `"rating" in place` tests whether `"rating"` is a **key**. Use that rule to predict
the last two results below.


In [ ]:
print(place.get("category"))
print(place.get("notes"))                       # missing, so None
print(place.get("notes", "no notes written"))   # missing, so the fallback

print("is 'rating' a key?", "rating" in place)  # test whether the key exists
print("is 'love it' a key?", "love it" in place)


### Convert a row to a dictionary and read nested dictionaries

`places.iloc[0]` selects the first row as a pandas **Series**, a one-dimensional collection of
labelled values. `.to_dict()` converts it to a dictionary with column names as keys and that row's
values as values.

A dictionary can also hold other dictionaries. In the GeoJSON example below, `feature` contains:

- `properties`, a dictionary with the place's name, category, and rating.
- `geometry`, a dictionary describing a point and its coordinates.

Read `feature["properties"]["name"]` from left to right: first select the `properties` dictionary,
then look up its `name` key. The coordinates are a list inside `geometry`. How would you select
the first number in that list?


In [ ]:
print(places.iloc[0].to_dict())
print()

feature = {
    "type": "Feature",
    "properties": {"name": "Doe Library", "category": "reading", "rating": "love it"},
    "geometry": {"type": "Point", "coordinates": [-122.2592, 37.8724]},
}

print("name       :", feature["properties"]["name"])
print("coordinates:", feature["geometry"]["coordinates"])
print("longitude  :", feature["geometry"]["coordinates"][0])
print("elevation  :", feature["properties"].get("elevation", "not recorded"))


GeoJSON point coordinates use **longitude first, latitude second**. In this example,
`feature["geometry"]["coordinates"][0]` returns the longitude, `-122.2592`.

Our CSV lists `lat` before `lon`, so pay attention to the order when constructing coordinates.
Latitude must be between `-90` and `90`; a latitude of `-122.2592` would indicate that something
is wrong. Range checks can catch some coordinate errors, although not every swap puts a value
outside the allowed range.


### Count categories and ratings together

`.value_counts()` counts the values in one column. `pd.crosstab()` counts combinations from two
columns. Here, its rows are categories and its columns are ratings.

Each cell tells us how many places have that category **and** that rating. For example, the
`food` row and `love it` column count food places rated `"love it"`.

Run the cell. What should all the counts add up to?


In [ ]:
pd.crosstab(places["category"], places["rating"])


The counts add up to fifteen in the supplied data, with one count per place. The `food` row has
three places rated `"love it"` and one rated `"just okay"`. All three `relaxing` places are rated
`"just okay"`.

The ratings in this file are teaching examples. The table describes those sample records.

The rating columns are in alphabetical order. Here, that happens to match their intended order:
`"just okay"`, `"like it"`, `"love it"`. Next we will specify that order explicitly.


### Compare text labels

`.max()` on a text column returns the value that comes last in string order. For these lowercase
rating labels, that is alphabetical order. It does not interpret what the ratings mean.

Which label will `places["rating"].max()` return?


In [ ]:
places["rating"].max()


The result is `"love it"`, which comes last alphabetically and is also our highest rating.
Those two orders do not always agree. If the labels were `"poor"`, `"fair"`, and `"excellent"`,
the last one alphabetically would be `"poor"`.

### Specify the order of categories

`pd.Categorical()` lets us define the rating order. The `categories` list gives the levels from
lowest to highest, and `ordered=True` tells pandas to use that order for comparisons and `.max()`.

The next cell defines the order, then counts ratings above `"like it"`. It also renames the levels
to `"poor"`, `"fair"`, and `"excellent"` while keeping their order. Compare the highest ordered
rating with the last label alphabetically.


In [ ]:
places["rating"] = pd.Categorical(places["rating"],
                                  categories=["just okay", "like it", "love it"],
                                  ordered=True)

print("levels, lowest to highest:", list(places["rating"].cat.categories))
print("above 'like it':", (places["rating"] > "like it").sum(), "places")
print("highest rating:", places["rating"].max())

# Rename each level while preserving the specified order.
renamed = places["rating"].cat.rename_categories(["poor", "fair", "excellent"])
print("highest rating after renaming:", renamed.max())
print("last label alphabetically:", max(renamed.cat.categories))


The ordered maximum is `"love it"` before renaming and `"excellent"` afterward. The last renamed
label alphabetically is `"poor"`. The categorical column uses the order we specified instead
of the spelling of each label.

Setting an order does not define a numerical distance between levels, so `.mean()` is still
unsupported for the categorical column.

Check the `categories` list carefully: its sequence determines the order. A value in the data
that is absent from this list becomes missing when converted.


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ✏️ Your Turn: combine conditions with `|`

Show places that are rated `"love it"` **or** belong to the `"bookstore"` category. Create a mask
for each condition, combine them with `|`, and use the result to select rows.

Before running the filter, predict the count for the supplied data. There are seven favourites
and three bookstores, but two bookstores are already among the favourites. How many distinct
rows should the result contain?


In [ ]:
# Your code here


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Summary

| Task | Example |
|---|---|
| Look up a dictionary value | `place["name"]` |
| Build a dictionary from paired lists | `dict(zip(place_names, place_ratings))` |
| Inspect dictionary keys and values | `.keys()`, `.values()`, `.items()` |
| Supply a default for a missing key | `place.get("notes", "no notes written")` |
| Update a category count | `counts[category] = counts.get(category, 0) + 1` |
| Build a DataFrame from column lists | `pd.DataFrame({...})` |
| Read a CSV file | `pd.read_csv(path)` |
| Inspect column types | `places.dtypes` |
| Count values in a column | `places["rating"].value_counts()` |
| Filter rows with a mask | `places[is_favourite]` |
| Select by position or label | `favourites.iloc[1]`, `favourites.loc[2, "name"]` |
| Combine masks | `mask_a & mask_b`, `mask_a | mask_b` |
| Count matches | `mask.sum()` |
| Split or slice one string | `address.split(", ")`, `address[-5:]` |
| Split or slice every string in a column | `places["address"].str.split(", ")`, `places["address"].str[-5:]` |
| Find text matches | `places["address"].str.contains("Durant")` |
| Plot counts or coordinates | `px.bar(...)`, `px.scatter(...)` |
| Estimate a local distance | Convert coordinate differences to metres, then use the Pythagorean theorem |
| Sort rows | `places.sort_values("distance_m")` |
| Count combinations of two columns | `pd.crosstab(places["category"], places["rating"])` |
| Convert a row to a dictionary | `places.iloc[0].to_dict()` |
| Read a nested dictionary | `feature["properties"]["name"]` |
| Set an order for categories | `pd.Categorical(values, categories=levels, ordered=True)` |

Next week, you will combine your team's files and check the resulting table for missing values,
inconsistent labels, and other data problems.
